# Projet 7 — Prévision de la demande de vélos partagés

| | |
|---|---|
| **Dataset** | Bike Sharing Dataset |
| **Source** | UCI Machine Learning Repository |
| **Domaine** | Mobilité urbaine |

**Problématique** : Comment la météo, la saison et le calendrier influencent-ils la demande de vélos partagés ?

**Compétences évaluées** : Régression, agrégation temporelle, variables calendaires, visualisation temporelle, comparaison de modèles.

## Bloc 1 — Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import urllib.request
import zipfile
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Bibliotheques importees avec succes')

## Bloc 2 — Téléchargement et chargement des données

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
ZIP_PATH = "Bike-Sharing-Dataset.zip"
DATA_DIR = "bike_data"

if not os.path.exists(ZIP_PATH):
    print("Telechargement en cours...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Telechargement termine.")
else:
    print("Archive deja presente.")

if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    print("Extraction terminee.")

day_df  = pd.read_csv(os.path.join(DATA_DIR, "day.csv"))
hour_df = pd.read_csv(os.path.join(DATA_DIR, "hour.csv"))

print(f"day.csv  : {day_df.shape[0]} lignes x {day_df.shape[1]} colonnes")
print(f"hour.csv : {hour_df.shape[0]} lignes x {hour_df.shape[1]} colonnes")

## Bloc 3 — Exploration initiale des données

In [ ]:
print("=== APERCU day.csv ===")
print(f"Dimensions : {day_df.shape}")
print()
display(day_df.head(6))

print("\n=== TYPES DE COLONNES ===")
print(day_df.dtypes)

print("\n=== COLONNES ET SIGNIFICATION ===")
col_info = {
    'instant'   : 'Identifiant de ligne',
    'dteday'    : 'Date',
    'season'    : 'Saison (1=Printemps, 2=Ete, 3=Automne, 4=Hiver)',
    'yr'        : 'Annee (0=2011, 1=2012)',
    'mnth'      : 'Mois (1-12)',
    'holiday'   : 'Jour ferie (0/1)',
    'weekday'   : 'Jour de semaine (0=Lundi)',
    'workingday': 'Jour ouvre (0/1)',
    'weathersit': 'Meteo (1=Clair, 2=Nuageux, 3=Pluie legere, 4=Pluie forte)',
    'temp'      : 'Temperature normalisee [0,1] => x41 pour degres Celsius',
    'atemp'     : 'Ressenti normalise [0,1] => x50',
    'hum'       : 'Humidite normalisee [0,1]',
    'windspeed' : 'Vitesse du vent normalisee [0,1]',
    'casual'    : 'Locations par utilisateurs non-inscrits',
    'registered': 'Locations par utilisateurs inscrits',
    'cnt'       : 'Total locations (cible = casual + registered)',
}
for col, desc in col_info.items():
    print(f"  {col:12s} : {desc}")

## Bloc 4 — Statistiques descriptives

In [ ]:
print("=== STATISTIQUES DESCRIPTIVES ===")
display(day_df.describe().round(3))

print("\n=== VALEURS MANQUANTES ===")
nb_missing = day_df.isnull().sum()
if nb_missing.sum() == 0:
    print("Aucune valeur manquante detectee — dataset propre.")
else:
    print(nb_missing[nb_missing > 0])

print("\n=== DOUBLONS ===")
print(f"Nombre de doublons : {day_df.duplicated().sum()}")

## Bloc 5 — Conversion de `dteday` en datetime (Action 3)

In [ ]:
day_df['dteday'] = pd.to_datetime(day_df['dteday'])

print(f"Type apres conversion : {day_df['dteday'].dtype}")
print(f"Date de debut : {day_df['dteday'].min().date()}")
print(f"Date de fin   : {day_df['dteday'].max().date()}")
duree = (day_df['dteday'].max() - day_df['dteday'].min()).days
print(f"Duree couverte : {duree} jours ({duree/365:.1f} ans)")

## Bloc 6 — Vérification : casual + registered = cnt (Action 6)

In [ ]:
day_df['_check'] = day_df['casual'] + day_df['registered']
is_valid = (day_df['_check'] == day_df['cnt']).all()

print(f"casual + registered == cnt pour toutes les lignes : {is_valid}")
print()
print("Verification sur les 8 premieres lignes :")
display(day_df[['dteday', 'casual', 'registered', '_check', 'cnt']].head(8))

if is_valid:
    print("\nConclusion : la colonne 'cnt' est la somme exacte de 'casual' et 'registered'.")
    print("=> Ces deux colonnes doivent etre EXCLUES des features pour eviter le data leakage.")

day_df.drop('_check', axis=1, inplace=True)

## Bloc 7 — Création des variables calendaires (Action 5)

In [ ]:
# --- Variable 1 : mois numerique ---
day_df['mois'] = day_df['dteday'].dt.month

# --- Variable 2 : numero de semaine ISO ---
day_df['semaine'] = day_df['dteday'].dt.isocalendar().week.astype(int)

# --- Variable 3 : est_weekend (samedi=6 ou dimanche=0) ---
day_df['est_weekend'] = day_df['weekday'].isin([0, 6]).astype(int)

# --- Variable 4 : periode de l'annee (saison calendaire personnalisee) ---
def get_periode(mois):
    if mois in [12, 1, 2]:
        return 'Hiver'
    elif mois in [3, 4, 5]:
        return 'Printemps'
    elif mois in [6, 7, 8]:
        return 'Ete'
    else:
        return 'Automne'

day_df['periode']     = day_df['mois'].apply(get_periode)
day_df['periode_num'] = day_df['periode'].map({'Hiver': 0, 'Printemps': 1, 'Ete': 2, 'Automne': 3})

print("Variables calendaires creees :")
print("  - mois        : numero de mois (1-12)")
print("  - semaine     : numero de semaine ISO (1-53)")
print("  - est_weekend : 1 si samedi/dimanche, 0 sinon")
print("  - periode     : Hiver / Printemps / Ete / Automne (texte)")
print("  - periode_num : encodage numerique de periode")

display(day_df[['dteday', 'weekday', 'mois', 'semaine', 'est_weekend', 'periode']].head(12))

## Bloc 8 — Figure 1 : Série temporelle de `cnt`

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(day_df['dteday'], day_df['cnt'],
        color='steelblue', linewidth=0.9, alpha=0.85, label='Demande quotidienne')
ax.fill_between(day_df['dteday'], day_df['cnt'], alpha=0.15, color='steelblue')

# Moyenne mobile 30 jours
rolling_mean = day_df.set_index('dteday')['cnt'].rolling(30).mean()
ax.plot(rolling_mean.index, rolling_mean.values,
        color='darkred', linewidth=2, label='Moyenne mobile 30j')

ax.set_title('Serie Temporelle — Demande quotidienne de velos (cnt)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Nombre de locations (cnt)')
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig('fig1_serie_temporelle.png', dpi=150)
plt.show()
print("Observations :")
print("  - Tendance haussiere nette entre 2011 et 2012")
print("  - Cycles saisonniers reguliers : pic en ete, creux en hiver")

## Bloc 9 — Figure 2 : Boxplot de la demande par saison

In [ ]:
season_labels = {1: 'Printemps', 2: 'Ete', 3: 'Automne', 4: 'Hiver'}
day_df['season_label'] = day_df['season'].map(season_labels)
order_saisons = ['Printemps', 'Ete', 'Automne', 'Hiver']
palette = ['#2ecc71', '#f39c12', '#e67e22', '#3498db']

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=day_df, x='season_label', y='cnt',
            order=order_saisons, palette=palette, ax=ax, width=0.5)
sns.stripplot(data=day_df, x='season_label', y='cnt',
              order=order_saisons, color='black', alpha=0.15, size=3, ax=ax)

ax.set_title('Distribution de la demande par saison', fontsize=14, fontweight='bold')
ax.set_xlabel('Saison')
ax.set_ylabel('Nombre de locations (cnt)')
plt.tight_layout()
plt.savefig('fig2_boxplot_saison.png', dpi=150)
plt.show()

print("Mediane par saison :")
for s in order_saisons:
    med = day_df[day_df['season_label'] == s]['cnt'].median()
    print(f"  {s:12s} : {med:.0f}")

## Bloc 10 — Figure 3 : Relation température / demande

In [ ]:
# Denormalisation : temp reel = temp_norm * 41 degres
day_df['temp_reel'] = day_df['temp'] * 41

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Scatter + courbe polynomiale ---
sc = axes[0].scatter(day_df['temp_reel'], day_df['cnt'],
                     c=day_df['season'], cmap='RdYlGn', alpha=0.55, s=25)
z = np.polyfit(day_df['temp_reel'], day_df['cnt'], 2)
p = np.poly1d(z)
x_line = np.linspace(day_df['temp_reel'].min(), day_df['temp_reel'].max(), 300)
axes[0].plot(x_line, p(x_line), color='darkred', linewidth=2.5, label='Tendance poly. deg. 2')
axes[0].set_title('Temperature vs Demande', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Temperature reelle (degres Celsius)')
axes[0].set_ylabel('cnt')
axes[0].legend()
plt.colorbar(sc, ax=axes[0], label='Saison')

# --- Heatmap de correlation ---
meteo_cols = ['temp_reel', 'hum', 'windspeed', 'weathersit', 'cnt']
corr = day_df[meteo_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Correlation meteo / cnt', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig3_temperature_demande.png', dpi=150)
plt.show()

corr_temp = day_df[['temp_reel', 'cnt']].corr().iloc[0, 1]
corr_hum  = day_df[['hum',      'cnt']].corr().iloc[0, 1]
corr_wind = day_df[['windspeed','cnt']].corr().iloc[0, 1]
print(f"Correlation temp / cnt     : {corr_temp:.3f}  => relation positive forte")
print(f"Correlation humidite / cnt : {corr_hum:.3f}  => relation negative moderee")
print(f"Correlation vent / cnt     : {corr_wind:.3f}  => relation negative faible")

## Bloc 11 — Analyse des tendances (Actions 4 & 10)

In [ ]:
weekday_labels = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
weather_map    = {1: 'Clair', 2: 'Nuageux', 3: 'Pluie legere', 4: 'Pluie forte'}
day_df['weather_label'] = day_df['weathersit'].map(weather_map)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Par mois
monthly = day_df.groupby('mois')['cnt'].mean()
axes[0, 0].bar(monthly.index, monthly.values,
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, 12)))
axes[0, 0].set_title('Demande moyenne par mois', fontweight='bold')
axes[0, 0].set_xlabel('Mois')
axes[0, 0].set_ylabel('cnt moyen')
axes[0, 0].set_xticks(range(1, 13))

# Par saison
seasonal = day_df.groupby('season_label')['cnt'].mean().reindex(order_saisons)
axes[0, 1].bar(seasonal.index, seasonal.values,
               color=['#2ecc71', '#f39c12', '#e67e22', '#3498db'])
axes[0, 1].set_title('Demande moyenne par saison', fontweight='bold')
axes[0, 1].set_xlabel('Saison')
axes[0, 1].set_ylabel('cnt moyen')

# Par jour de semaine
wd_avg = day_df.groupby('weekday')['cnt'].mean()
bar_colors = ['#e74c3c' if i in [0, 6] else 'steelblue' for i in wd_avg.index]
axes[1, 0].bar([weekday_labels[i] for i in wd_avg.index], wd_avg.values, color=bar_colors)
axes[1, 0].set_title('Demande moyenne par jour de semaine\n(rouge = week-end)', fontweight='bold')
axes[1, 0].set_xlabel('Jour')
axes[1, 0].set_ylabel('cnt moyen')

# Par condition meteo
weather_avg = day_df.groupby('weather_label')['cnt'].mean()
worder = ['Clair', 'Nuageux', 'Pluie legere']
worder = [w for w in worder if w in weather_avg.index]
axes[1, 1].bar(weather_avg.reindex(worder).index,
               weather_avg.reindex(worder).values,
               color=['#f1c40f', '#95a5a6', '#3498db'])
axes[1, 1].set_title('Demande moyenne par condition meteo', fontweight='bold')
axes[1, 1].set_xlabel('Condition meteo')
axes[1, 1].set_ylabel('cnt moyen')

plt.suptitle('Analyse des Tendances de la Demande de Velos', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_tendances.png', dpi=150)
plt.show()

## Bloc 12 — Stratégie de split : justification et implémentation (Action 7)

### Choix : Split Temporel

#### Pourquoi PAS le split aléatoire ?
- Les données sont une **série temporelle** avec des corrélations temporelles et une tendance croissante.
- Un split aléatoire permettrait au modèle d'utiliser des données **futures** pour prédire le passé → **fuite de données (data leakage)**.
- Cela suroptimiserait artificiellement les métriques d'évaluation.

#### Pourquoi le split temporel ?
- On entraîne sur les **données passées** et on teste sur les **données futures** : c'est le scénario réel de production.
- La date de coupure au **1er juillet 2012** donne ~80% train / ~20% test, ce qui est un ratio standard.
- On préserve ainsi la structure temporelle des données.

#### Justification de la date de coupure
- Les données couvrent 2011-01-01 → 2012-12-31 (2 ans).
- On choisit `2012-07-01` pour avoir suffisamment de données de test couvrant l'été 2012 (période de forte demande).

In [ ]:
CUTOFF_DATE = pd.Timestamp('2012-07-01')

train_df = day_df[day_df['dteday'] <  CUTOFF_DATE].copy()
test_df  = day_df[day_df['dteday'] >= CUTOFF_DATE].copy()

pct_train = 100 * len(train_df) / len(day_df)
pct_test  = 100 * len(test_df)  / len(day_df)

print("=" * 55)
print("STRATEGIE : SPLIT TEMPOREL")
print("=" * 55)
print(f"Date de coupure    : {CUTOFF_DATE.date()}")
print(f"Ensemble TRAIN     : {len(train_df)} jours ({pct_train:.1f}%)")
print(f"  de {train_df['dteday'].min().date()} a {train_df['dteday'].max().date()}")
print(f"Ensemble TEST      : {len(test_df)} jours ({pct_test:.1f}%)")
print(f"  de {test_df['dteday'].min().date()} a {test_df['dteday'].max().date()}")

# Visualisation du split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df['dteday'], train_df['cnt'], color='steelblue', linewidth=1, label=f'Train ({len(train_df)} jours)')
ax.plot(test_df['dteday'],  test_df['cnt'],  color='coral',     linewidth=1, label=f'Test ({len(test_df)} jours)')
ax.axvline(x=CUTOFF_DATE, color='black', linestyle='--', linewidth=2, label='Coupure 2012-07-01')
ax.set_title('Split Temporel — Train vs Test', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('cnt')
ax.legend()
plt.tight_layout()
plt.savefig('fig5_split_temporel.png', dpi=150)
plt.show()

## Bloc 13 — Préparation des features (Action 2)

In [ ]:
# Colonnes a EXCLURE et pourquoi
exclude_cols = {
    'instant'     : 'Identifiant de ligne, pas informatif',
    'dteday'      : 'Date brute (deja encodee via mois, semaine, yr...)',
    'cnt'         : 'Variable CIBLE',
    'casual'      : 'Composante de cnt => data leakage',
    'registered'  : 'Composante de cnt => data leakage',
    'season_label': 'Version texte de season (deja numerique)',
    'weather_label': 'Version texte de weathersit (deja numerique)',
    'periode'     : 'Version texte de periode_num',
    'temp_reel'   : 'Redondant avec temp (simple scaling)',
}

print("Colonnes exclues :")
for col, raison in exclude_cols.items():
    marker = "x" if col in day_df.columns else "-"
    print(f"  [{marker}] {col:15s} : {raison}")

feature_cols = [c for c in day_df.columns if c not in exclude_cols]

print(f"\nFeatures retenues ({len(feature_cols)}) :")
for f in feature_cols:
    print(f"  + {f}")

X_train = train_df[feature_cols]
y_train = train_df['cnt']
X_test  = test_df[feature_cols]
y_test  = test_df['cnt']

# Normalisation (necessaire pour Linear Regression et Ridge)
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

print(f"\nX_train : {X_train.shape}  |  X_test : {X_test.shape}")

## Bloc 14 — Entraînement et évaluation des 4 modèles (Actions 8 & 9)

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Entraine un modele et retourne ses metriques + predictions."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    print(f"{'─'*45}")
    print(f"Modele  : {name}")
    print(f"  MAE   : {mae:.2f}")
    print(f"  RMSE  : {rmse:.2f}")
    print(f"  R2    : {r2:.4f}")
    return {'Modele': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'y_pred': y_pred, 'model': model}

results = []

# --- 1. Regression Lineaire ---
print("\n[1/4] Regression Lineaire")
lr = LinearRegression()
results.append(evaluate_model("Regression Lineaire", lr,
                              X_train_sc, X_test_sc, y_train, y_test))

# --- 2. Ridge Regression ---
print("\n[2/4] Ridge (regularisation L2, alpha=1.0)")
ridge = Ridge(alpha=1.0)
results.append(evaluate_model("Ridge (alpha=1.0)", ridge,
                              X_train_sc, X_test_sc, y_train, y_test))

# --- 3. Random Forest ---
print("\n[3/4] Random Forest Regressor")
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                           min_samples_leaf=2, random_state=42, n_jobs=-1)
results.append(evaluate_model("Random Forest", rf,
                              X_train, X_test, y_train, y_test))

# --- 4. Gradient Boosting ---
print("\n[4/4] Gradient Boosting Regressor")
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.08,
                               max_depth=4, subsample=0.8, random_state=42)
results.append(evaluate_model("Gradient Boosting", gb,
                              X_train, X_test, y_train, y_test))

## Bloc 15 — Comparaison des modèles

In [ ]:
metrics_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_pred', 'model']}
                             for r in results])
metrics_df = metrics_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("=== TABLEAU COMPARATIF ===")
display(metrics_df[['Modele', 'MAE', 'RMSE', 'R2']].round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    vals = [r[metric] for r in results]
    names = [r['Modele'] for r in results]
    bars = axes[i].bar(names, vals, color=colors, edgecolor='white', linewidth=0.8)
    axes[i].set_title(f'Comparaison — {metric}', fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width() / 2.,
                     bar.get_height() * 1.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig6_comparaison_modeles.png', dpi=150)
plt.show()

# Meilleur modele
best_idx    = max(range(len(results)), key=lambda i: results[i]['R2'])
best_result = results[best_idx]
print(f"\nMeilleur modele : {best_result['Modele']}")
print(f"  R2   = {best_result['R2']:.4f}  => explique {best_result['R2']*100:.1f}% de la variance")
print(f"  MAE  = {best_result['MAE']:.2f} velos/jour")
print(f"  RMSE = {best_result['RMSE']:.2f} velos/jour")

## Bloc 16 — Figure 4 : Analyse des résidus du meilleur modèle

In [ ]:
y_pred_best = best_result['y_pred']
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Residus vs predictions
axes[0].scatter(y_pred_best, residuals, alpha=0.45, color='steelblue', s=22)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title(f'Residus vs Predictions\n({best_result["Modele"]})', fontweight='bold')
axes[0].set_xlabel('Valeurs predites')
axes[0].set_ylabel('Residus')

# Distribution des residus
axes[1].hist(residuals, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(0,   color='red',   linestyle='--', linewidth=1.5, label='0')
axes[1].axvline(np.mean(residuals), color='navy', linestyle='-', linewidth=1.5,
                label=f'Moyenne={np.mean(residuals):.0f}')
axes[1].set_title('Distribution des Residus', fontweight='bold')
axes[1].set_xlabel('Residu')
axes[1].set_ylabel('Frequence')
axes[1].legend()

# Reel vs predit
min_v = min(y_test.min(), y_pred_best.min())
max_v = max(y_test.max(), y_pred_best.max())
axes[2].scatter(y_test.values, y_pred_best, alpha=0.45, color='seagreen', s=22)
axes[2].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1.8, label='Prediction parfaite')
axes[2].set_title('Valeurs Reelles vs Predites', fontweight='bold')
axes[2].set_xlabel('Valeur reelle')
axes[2].set_ylabel('Valeur predite')
axes[2].legend()

plt.tight_layout()
plt.savefig('fig7_residus.png', dpi=150)
plt.show()

print(f"Residus — Moyenne : {np.mean(residuals):.2f} | Std : {np.std(residuals):.2f}")
print(f"95% des residus dans l'intervalle [{np.percentile(residuals,2.5):.0f}, {np.percentile(residuals,97.5):.0f}]")

## Bloc 17 — Importance des variables (Action 10)

In [ ]:
best_model = best_result['model']

if hasattr(best_model, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'feature'   : feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_imp = ['#e74c3c' if v > 0.08 else '#3498db' for v in imp_df['importance']]
    ax.barh(imp_df['feature'], imp_df['importance'], color=colors_imp, edgecolor='white')
    ax.set_title(f'Importance des Variables — {best_result["Modele"]}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance (Gini / gain moyen)')
    plt.tight_layout()
    plt.savefig('fig8_feature_importance.png', dpi=150)
    plt.show()

    print("TOP 5 features (rouge dans le graphe) :")
    top5 = imp_df.sort_values('importance', ascending=False).head(5)
    for _, row in top5.iterrows():
        print(f"  {row['feature']:15s} : {row['importance']:.4f}")
else:
    print(f"Le modele {best_result['Modele']} n'expose pas de feature_importances_.")
    print("Utilisez les coefficients (lr.coef_) pour la regression lineaire.")

## Bloc 18 — Interprétation des résultats (Action 10)

In [ ]:
print("=" * 62)
print("INTERPRETATION DES FACTEURS - DEMANDE DE VELOS PARTAGES")
print("=" * 62)

print("FACTEURS QUI AUGMENTENT LA DEMANDE :")
print("  [+] Temperature (temp)      : Plus il fait chaud => plus de cyclistes")
print("  [+] Annee (yr=1=2012)       : Croissance du service entre 2011 et 2012")
print("  [+] Meteo claire (wthr=1)   : Beau temps => forte demande")
print("  [+] Ete / Automne           : Saisons les plus actives")
print("  [+] Jour ouvre (workingday) : Les navetteurs utilisent le service en semaine")
print()
print("FACTEURS QUI DIMINUENT LA DEMANDE :")
print("  [-] Vent fort (windspeed)   : Decourage les sorties a velo")
print("  [-] Humidite elevee (hum)   : Sensation desagreable, moins de sorties")
print("  [-] Pluie (weathersit >= 3) : Fort impact negatif sur la demande")
print("  [-] Hiver (season=4)        : Temperatures basses => forte baisse")
print()
print("EFFETS NUANCES :")
print("  [~] Week-end (est_weekend)  : Profil different (loisirs vs trajet)")
print("  [~] Jours feries (holiday)  : Baisse pour les navetteurs, hausse touristes")
print("  [~] Mois                    : Reflete la saisonnalite (juin-sept = pics)")

print("=" * 62)
print("RESUME FINAL DES PERFORMANCES")
print("=" * 62)
summary = metrics_df[['Modele', 'MAE', 'RMSE', 'R2']].round(4)
display(summary)

print("CONCLUSION :")
print(f"  Le meilleur modele explique {'{:.1f}%'.format(best_result['R2']*100)} de la variance.")
print(f"  MAE  = {best_result['MAE']:.0f} velos/jour")
print(f"  RMSE = {best_result['RMSE']:.0f} velos/jour")
print("  Il surpasse la regression lineaire car il capture les")
print("  relations non-lineaires (temperature, saisonnalite).")